## Aula 08 - 20/10/2026

Na Aula 07 filtramos um único DataFrame (o espaço amostral Ω) com máscaras booleanas. Sistemas reais, porém, não guardam dados em uma tabela única: eles os distribuem em múltiplas entidades que se relacionam por **chaves** (identificadores únicos). Cruzar essas entidades exige a **Álgebra Relacional** de Codd (1970): união (∪), interseção (∩), diferença (−), produto cartesiano (×), seleção (σ) e projeção (π).

As operações de conjuntos foram estudadas em seções passadas, visto que no momento estudaremos as operações ainda não definidas em nossos materiais. A primeira operação a ser definida serpa a operação de Seleção ou Select.

### Select - σ

A operação de seleção irá, a partir de uma tabela original, criar uma nova tabela que satisfaça uma certa propriedade de algum atributo em particular (linhas).

### Project - π

A operação de projeção irá, a partir de uma tabela original, uma nova tabela composta por determinadas colunas desta.

### Join

O Join refere-se à operações dos conjuntos. Ela é realizada em função de um atributo em comum entre as tabelas. A operação irá formar um subconjunto do produto cartesiano de todas as n-uplas - ou linhas - da primeira relação com todas as k-uplas - ou linhas - da segunda relação, resultando em um conjunto com (n+k)-uplas, selecionando destre estes os que possuem atributo comum com o mesmo valor.

Tópcio a analisar: SQL e demais atribuições em nosso [Livro texto](https://animaedu-my.sharepoint.com/:b:/r/personal/ciro_chaves_ulife_com_br/Documents/Matem%C3%A1tica%20Computacional%20Aplicada%202026-2/Livro%20texto/Fundamentos%20Matem%C3%A1ticos%20para%20Ci%C3%AAncias%20da%20Computa%C3%A7%C3%A3o.pdf?d=w983aa8dbb1e140b5818df8d46dedb708&csf=1&web=1&e=zjv0uQ).

*Lembre-se de que só com o seu e-mail institucional você pode abrir a URL acima.*

### Live coding 1 - [Slide da Aula #8 PRÁTICA UNIFG](https://animaedu-my.sharepoint.com/:p:/r/personal/ciro_chaves_ulife_com_br/Documents/Matem%C3%A1tica%20Computacional%20Aplicada%202026-2/Slides%20-%20Pr%C3%A1tica/Aula%20%238%20PR%C3%81TICA%20UNIFG%20-%20%C3%81lgebra%20Relacional%20Pandas.pptx?d=wf8c3ef7a12ca4799acf6b610f4fc658b&csf=1&web=1&e=2LAcvU)

## O cenário: duas fontes de verdade que divergem

Toda operação de infraestrutura mantém, no mínimo, **dois registros sobre os mesmos ativos**:

- O **inventário** é a planilha de cadastro de cada servidor provisionado, que recebe um `id_srv` e um IP no momento do cadastro.
- O **monitoramento** é a tela do software de monitoramento: Ela diz o que funciona no momento, de acordo com o seu `status`.

Essas tabelas são alimentadas por rotinas diferentes — o provisionamento instala o sistema operacional e cadastra a máquina no inventário; a instalação do agente de monitoramento é outra etapa, de outra equipe, em outro dia.

E nenhuma das duas tabelas, sozinha, responde às perguntas que importam:

1. **Cobertura** — quais servidores cadastrados estão sem monitoramento? (pontos cegos da operação);
2. **Órfãos** — quais máquinas reportam métricas sem constar no inventário? (shadow IT);
3. **Divergência total** — quantos ativos estão fora da interseção, somando os dois lados?

Só o cruzamento relacional responde. Nos próximos blocos, adotamos a convenção: **A = inventário (`servidores`)** e **B = monitoramento (`monitorados`)**.

In [39]:
import pandas as pd

# Conjunto A: Inventário de Servidores
servidores = pd.DataFrame(
    {
        "id_srv": ["SRV_01", "SRV_02", "SRV_03", "SRV_04"],
        "ip": ["10.0.0.1", "10.0.0.2", "10.0.0.3", "10.0.0.4"],
    }
)

# Conjunto B: Monitoramento Ativo
monitorados = pd.DataFrame(
    {
        "id_srv": ["SRV_02", "SRV_03", "SRV_05"],
        "status": ["ONLINE", "ONLINE", "ALERTA"],
    }
)


In [40]:
servidores.head()

,id_srv,ip
0,SRV_01,10.0.0.1
1,SRV_02,10.0.0.2
2,SRV_03,10.0.0.3
3,SRV_04,10.0.0.4


In [41]:
monitorados.head()

,id_srv,status
0,SRV_02,ONLINE
1,SRV_03,ONLINE
2,SRV_05,ALERTA


Note a assimetria deliberada entre os conjuntos: `SRV_01` e `SRV_04` existem no inventário (A) mas **não** estão no monitoramento (B); `SRV_05` está em B mas **não** consta em A. Essa configuração permite observar o comportamento de cada tipo de join:

- **Inner (A ∩ B):** Retorna apenas as chaves presentes nas duas tabelas → `SRV_02`, `SRV_03`.
- **Outer (A ∪ B):** Retorna todas as chaves; as ausências viram `NaN`.
- **Left (domínio de A):** mantém 100% da tabela A e agrega dados de B quando existirem.
- **Cross (A × B):** todos os pares ordenados possíveis, |A| · |B| linhas.


In [42]:
df_inner = pd.merge(
    servidores,
    monitorados,
    on="id_srv",
    how="inner",
)
print(df_inner)

print('\n\n')

df_outer = pd.merge(
    servidores,
    monitorados,
    on="id_srv",
    how="outer",
)
print(df_outer)


   id_srv        ip  status
0  SRV_02  10.0.0.2  ONLINE
1  SRV_03  10.0.0.3  ONLINE



   id_srv        ip  status
0  SRV_01  10.0.0.1     NaN
1  SRV_02  10.0.0.2  ONLINE
2  SRV_03  10.0.0.3  ONLINE
3  SRV_04  10.0.0.4     NaN
4  SRV_05       NaN  ALERTA


**Problema de auditoria:** quais servidores estão cadastrados no inventário (A) mas **não** possuem monitoramento ativo (B)?

Matematicamente, é a diferença **A − B**, conhecida em bancos de dados como **Anti-Join**. O Pandas não tem um método `anti_join`, então a solução vetorizada combina:

1. `.isin()` — produz um vetor booleano de pertinência (cada chave de A está em B?);
2. `~` — operador lógico de negação (NOT), que inverte a máscara e filtra a diferença.


In [43]:
mascara = servidores["id_srv"].isin(monitorados["id_srv"])
print(mascara.tolist())
print('\n')

sem_agente = servidores[~mascara]
print(sem_agente)


[False, True, True, False]


   id_srv        ip
0  SRV_01  10.0.0.1
3  SRV_04  10.0.0.4


O parâmetro `indicator=True` adiciona ao resultado a coluna especial `_merge`, que classifica matematicamente a origem de cada linha em uma única operação:

- `both` — pertence à interseção (A ∩ B);
- `left_only` — exclusivo da esquerda (A − B);
- `right_only` — exclusivo da direita (B − A).

É a ferramenta padrão para auditorias de consistência entre sistemas: em vez de três queries, um único Outer Merge rotulado.

In [44]:
auditoria = pd.merge(
    servidores,
    monitorados,
    on="id_srv",
    how="outer",
    indicator=True,
)

print(auditoria)

   id_srv        ip  status      _merge
0  SRV_01  10.0.0.1     NaN   left_only
1  SRV_02  10.0.0.2  ONLINE        both
2  SRV_03  10.0.0.3  ONLINE        both
3  SRV_04  10.0.0.4     NaN   left_only
4  SRV_05       NaN  ALERTA  right_only


*Observação importante:*

Conjuntos matemáticos não admitem repetição; DataFrames, sim. Antes de operar com cardinalidades de chaves, reduza a tabela (ou coluna) a um conjunto próprio:

```python
df_unico = df.drop_duplicates()
```

Esse detalhe será decisivo na Missão 03 do desafio: sem `drop_duplicates()`, contagens de chaves ficam infladas por eventos repetidos.

### Exemplo Aplicativo com o DataSet Titanic

Em 1912, o Titanic recebeu passageiros em três portos: **Southampton** (Inglaterra), **Cherbourg** (França) e **Queenstown** (Irlanda). O dataset público, porém, registra o embarque apenas pela sigla — `'S'`, `'C'` ou `'Q'` — na coluna `Embarked`.

A sigla é uma **chave sem semântica**: sozinha, ela não diz cidade nem país. Qualquer análise geográfica — tarifa média por país de embarque, taxa de sobrevivência por porto — exige primeiro traduzir essa chave.

É para isso que existe a **tabela dimensional**: uma tabela pequena, mantida à parte, que mapeia cada sigla para seus atributos descritivos (`Porto`, `Pais`). O Join é a operação que enriquece 891 registros de passageiros com o dicionário de portos — sem copiar nenhum dado manualmente.

A pergunta arquitetural que guiará o experimento: **quando enriquecemos a tabela dos passageiros com os portos da Europa Continental, o que acontece com quem embarcou em Southampton?** A resposta depende inteiramente da escolha entre `inner` e `left`. Vamos ver no exemplo:

In [45]:
url = (
    "https://raw.githubusercontent.com/"
    "datasciencedojo/datasets/master/titanic.csv"
)

df_titanic = pd.read_csv(url)
print('Dimensões do DataFrame Titanic: ')
print(df_titanic.shape)
print('\n')
df_titanic.head()

Dimensões do DataFrame Titanic: 
(891, 12)




,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [46]:
print('Coluna Embarked:\n')
print(df_titanic['Embarked'])

Coluna Embarked:

0      S
1      C
2      S
3      S
4      S
      ..
886    S
887    S
888    S
889    C
890    Q
Name: Embarked, Length: 891, dtype: str


In [47]:
portos_europa = pd.DataFrame(
    {
        "Embarked": ["C", "Q"],
        "Porto": ["Cherbourg", "Queenstown"],
        "Pais": ["França", "Irlanda"],
    }
)

print(portos_europa)

  Embarked       Porto     Pais
0        C   Cherbourg   França
1        Q  Queenstown  Irlanda


In [48]:
df_inner = pd.merge(
    df_titanic,
    portos_europa,
    on="Embarked",
    how="inner",
)
print('Qtd de elementos no Inner Join:', df_inner.shape[0])

df_left = pd.merge(
    df_titanic,
    portos_europa,
    on="Embarked",
    how="left",
)
print('Quantidade de elementos no Left Join:', df_left.shape[0])


Qtd de elementos no Inner Join: 245
Quantidade de elementos no Left Join: 891


- **891** linhas no `df_left` — preserva 100% do espaço amostral Ω; os passageiros de `'S'` (Southampton) recebem `NaN` nas colunas `Porto` e `Pais`.
- **245** linhas no `df_inner` — restam apenas quem embarcou em Cherbourg (168) ou Queenstown (77).
- **646** passageiros foram descartados pelo Inner Join: 644 de Southampton (`'S'`) + 2 registros com `Embarked` nulo.

Regra de engenharia: se a ausência do par é um dado legítimo do domínio (o passageiro simplesmente não embarcou na Europa Continental), use `left`; se a ausência indica registro inválido para a análise, use `inner`.

In [49]:
df_inner.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Porto,Pais
0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cherbourg,França
1,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,Queenstown,Irlanda
2,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,Cherbourg,França
3,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.1250,NaN,Q,Queenstown,Irlanda
4,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C,Cherbourg,França


In [50]:
df_left.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Porto,Pais
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,NaN,NaN
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cherbourg,França
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,NaN,NaN
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,NaN,NaN
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,NaN,NaN


In [51]:
print('Mostrando apenas colunas de interesse do Inner Join: ')
print(df_inner[['Name', 'Embarked', 'Porto', 'Pais']])

Mostrando apenas colunas de interesse do Inner Join: 
                                                  Name Embarked       Porto  \
0    Cumings, Mrs. John Bradley (Florence Briggs Th...        C   Cherbourg   
1                                     Moran, Mr. James        Q  Queenstown   
2                  Nasser, Mrs. Nicholas (Adele Achem)        C   Cherbourg   
3                                 Rice, Master. Eugene        Q  Queenstown   
4                              Masselmani, Mrs. Fatima        C   Cherbourg   
..                                                 ...      ...         ...   
240                   Najib, Miss. Adele Kiamie "Jane"        C   Cherbourg   
241      Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)        C   Cherbourg   
242               Rice, Mrs. William (Margaret Norton)        Q  Queenstown   
243                              Behr, Mr. Karl Howell        C   Cherbourg   
244                                Dooley, Mr. Patrick        Q  Queenstown  

In [52]:
print('Mostrando apenas colunas de interesse do Left Join: ')
print(df_left[['Name', 'Embarked', 'Porto', 'Pais']])

Mostrando apenas colunas de interesse do Left Join: 
                                                  Name Embarked       Porto  \
0                              Braund, Mr. Owen Harris        S         NaN   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...        C   Cherbourg   
2                               Heikkinen, Miss. Laina        S         NaN   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)        S         NaN   
4                             Allen, Mr. William Henry        S         NaN   
..                                                 ...      ...         ...   
886                              Montvila, Rev. Juozas        S         NaN   
887                       Graham, Miss. Margaret Edith        S         NaN   
888           Johnston, Miss. Catherine Helen "Carrie"        S         NaN   
889                              Behr, Mr. Karl Howell        C   Cherbourg   
890                                Dooley, Mr. Patrick        Q  Queenstown   

### Exercício Prático

## Cenário

Vocês foram contratados como analistas de dados para auditar o controle de acesso físico de um data center. O prédio tem salas restritas (`Core`, onde ficam os servidores; `Cofre`, onde ficam as fitas de backup; e `Lab`, o laboratório de manutenção), e a diretoria de segurança suspeita de **tentativas de acesso não autorizado** na última noite.

Dois sistemas independentes registram a operação:

**Tabela 1 — `credenciais` (o cadastro de quem pode entrar)**

| Coluna | Significado |
| --- | --- |
| `id_user` | número único que identifica cada pessoa credenciada |
| `nome` | nome do funcionário |
| `nivel` | perfil de acesso (`Admin`, `Dev`, `SecOps`, `Guest`) |

**Tabela 2 — `logs_catraca` (o registro de quem entrou de fato)**

| Coluna | Significado |
| --- | --- |
| `id_log` | número sequencial do evento registrado na catraca |
| `id_user` | o ID lido pelo crachá naquela passagem |
| `sala` | sala que a pessoa tentou acessar (`Core`, `Lab` ou `Cofre`) |

### Três armadilhas plantadas nos dados (leia antes de codar)

1. **Eventos ≠ pessoas.** A catraca registra *passagens*, não pessoas. Um funcionário que entra, sai para almoçar e volta aparece **duas vezes** nos logs — e isso é legítimo. Se você contar linhas de um join achando que está contando pessoas, vai reportar o número errado para a diretoria.
2. **Existem IDs fantasma.** Há passagens nos logs cujo `id_user` **não existe** no cadastro de credenciais. É exatamente isso que a auditoria precisa encontrar — mas eles vão "sumir" silenciosamente se você usar o join errado, ou "poluir" sua contagem de pessoas se você esquecer de separá-los.
3. **Existem credenciais ociosas.** Há pessoas credenciadas que **não apareceram** para trabalhar. Elas não são um problema de segurança, mas precisam entrar no relatório para a conta fechar.

## Suas missões

### Missão 01 — Relatório de acessos válidos (Interseção)

O time de segurança precisa de um relatório legível: **cada passagem válida pela catraca, com o nome e o nível de acesso de quem passou.**

- Cruze `logs_catraca` com `credenciais` de forma que cada log seja associado à credencial correspondente, e que passagens sem credencial válida sejam descartadas automaticamente.
- **Pergunta que você deve responder no notebook:** quantos *eventos de log* foram validados? E quantas *pessoas distintas* entraram no prédio? (Se os dois números forem diferentes, explique em uma frase por quê.)

### Missão 02 — Alerta de intrusão (Diferença B − A)

A diretoria quer a lista exata das **tentativas de acesso sem credencial**: quais eventos, quais IDs e quais salas foram alvo.

- Isole apenas os registros de `logs_catraca` cujo `id_user` **não possui cadastro** em `credenciais`.
- **Restrição técnica:** é proibido usar laço `for`. Use o padrão vetorizado com `.isin()` + negação `~` visto no live coding.
- **Pergunta que você deve responder no notebook:** quais salas os intrusos tentaram acessar? (Observe se os alvos são salas críticas — isso mudaria o tom do seu relatório para a diretoria.)

### Missão 03 — Fechamento da auditoria (a conta tem que fechar)

Toda auditoria termina com uma conferência de totais: a prova de que nenhuma chave foi perdida nem contada duas vezes. Aqui vocês vão descobrir essa conferência **passo a passo** — não é preciso conhecer fórmula nenhuma de antemão.

- **Passo 1 — conjuntos de chaves.** Extraia as chaves únicas de cada tabela (lembre-se: conjuntos não têm repetição — use o método que "reduz a tabela a um conjunto próprio"). Anote: |K_A| e |K_B|.
- **Passo 2 — união e interseção.** Com um Outer Merge + `indicator=True` sobre as chaves únicas, conte quantas chaves existem no total (união) e quantas aparecem nas duas tabelas (interseção). Anote: |K_A ∪ K_B| e |K_A ∩ K_B|.
- **Passo 3 — os que ficaram de fora.** Conte diretamente, na coluna `_merge`, quantas chaves **não** são `both` (ou seja: ociosos + intrusos).
- **Passo 4 — a descoberta.** Compare o resultado do Passo 3 com os números do Passo 2. Que conta simples envolvendo união e interseção produz exatamente o mesmo valor? Escreva essa conta no notebook — ela é a sua ferramenta de conferência para qualquer auditoria futura.
- **Pergunta que você deve responder no notebook:** o total de chaves fora da interseção se decompõe em quantas credenciais ociosas + quantas tentativas de intrusão? Esse número bate com o que você encontrou na Missão 02?

## O que entregar (critério de conclusão)

1. Os três DataFrames resultantes (`acessos_validos`, `invasoes` e a tabela de auditoria da Missão 03) **impressos** no notebook;
2. As contagens anotadas em um comentário ou célula Markdown: |K_A|, |K_B|, |K_A ∪ K_B|, |K_A ∩ K_B| e o total de chaves fora da interseção, **acompanhadas da conta que você descobriu no Passo 4**;
3. As respostas das três perguntas em texto, no próprio notebook;
4. `git commit` + `git push` do `aula08_desafio.ipynb` no repositório da dupla.

## Dicas (consulte antes de pedir socorro)

- O Titanic mostrou a diferença entre `inner` e `left`: qual dos dois a Missão 01 pede? O que aconteceria com os intrusos se você escolhesse o outro?
- Na Missão 02, `mascara = logs["id_user"].isin(credenciais["id_user"])` responde "este log tem credencial?". O que você precisa é do **contrário** disso.
- Na Missão 03, contar `id_user` com `len()` direto na tabela de logs conta **eventos**, não chaves. O conjunto de chaves exige um passo a mais antes da contagem. E no Passo 4, experimente combinar os dois números do Passo 2 com uma subtração — o resultado deve ser igual ao do Passo 3.


In [53]:
import pandas as pd

credenciais = pd.DataFrame(
    {
        "id_user": [101, 102, 103, 104, 105, 106],
        "nome": ["Alice", "Bruno", "Carla", "Diego", "Elena", "Fabio"],
        "nivel": ["Admin", "Dev", "Dev", "SecOps", "Admin", "Guest"],
    }
)

logs_catraca = pd.DataFrame(
    {
        "id_log": [1, 2, 3, 4, 5, 6, 7],
        "id_user": [101, 103, 101, 999, 105, 888, 102],
        "sala": ["Core", "Lab", "Cofre", "Core", "Cofre", "Lab", "Lab"],
    }
)

print(credenciais)
print('\n')
print(logs_catraca)


   id_user   nome   nivel
0      101  Alice   Admin
1      102  Bruno     Dev
2      103  Carla     Dev
3      104  Diego  SecOps
4      105  Elena   Admin
5      106  Fabio   Guest


   id_log  id_user   sala
0       1      101   Core
1       2      103    Lab
2       3      101  Cofre
3       4      999   Core
4       5      105  Cofre
5       6      888    Lab
6       7      102    Lab
